# Dataset Notebook
### Samuel Ivan Sanchez Salazar

Notebook to preprocess, filter and clean datasets
## Imports

In [2]:
import numpy as np
import pandas as pd

## Load Raw Datasets

In [3]:
top5_players = pd.read_csv("../Datasets/Top5_League_Players_2017to2024_dataset.csv", delimiter=';', decimal=',')
player_valuations = pd.read_csv("../Datasets/player_valuations.csv")
players = pd.read_csv("../Datasets/players.csv")
transfers = pd.read_csv("../Datasets/transfers.csv")

## Merge, Filters, Grouping and Cleaning
Select only relevant columns

In [4]:
relevant_stats = [
    "league",
    "season",
    "team",
    "player",
    "nation_",
    "pos_",
    "age_",
    "born_",
    "Playing Time_Min",
    "Playing Time_90s",
    "Performance_Gls",
    "Performance_Ast",
    "Performance_G+A",
    "Performance_PK",
    "Performance_Fld",
    "Performance_Fls",
    "Performance_Int",
    "Performance_Off",
    "Performance_Recov",
    "Performance_TklW",
    "Tackles_Att 3rd",
    "Expected_xG",
    "Expected_npxG",
    "Expected_xAG",
    "Expected_xA",
    "Expected_G-xG",
    "Progression_PrgC",
    "Progression_PrgP",
    "Progression_PrgR",
    "Per 90 Minutes_Gls",
    "Per 90 Minutes_Ast",
    "Per 90 Minutes_G+A",
    "Per 90 Minutes_xG",
    "Per 90 Minutes_xAG",
    "Per 90 Minutes_npxG",
    "Standard_SoT%",
    "Standard_Sh/90",
    "Standard_G/Sh",
    "KP_",
    "PrgP_",
    "1/3_",
    "Pass Types_TB",
    "Total_Att",
    "Total_Cmp%",
    "Long_Cmp%",
    "Medium_Cmp%",
    "Short_Cmp%",
    "SCA_SCA90",
    "GCA_GCA90",
    "Err_",
    "Blocks_Blocks",
    "Take-Ons_Att",
    "Take-Ons_Succ%",
    "Carries_Carries",
    "Carries_PrgC",
    "Carries_Dis",
    "Touches_Touches",
    "Touches_Att Pen",
    "Aerial Duels_Won",
    "Aerial Duels_Won%",
    "Team Success_PPM",
    "Team Success_+/-90",
]
top5_players = top5_players[relevant_stats]

Standardize column with player's name for merging

In [5]:
top5_players['name'] = (top5_players['player']
                             .str.normalize('NFKD') 
                             .str.encode('ascii', errors='ignore') 
                             .str.decode('utf-8') 
                             .str.lower() 
                             .str.strip()) 

df_players_mini = players[['player_id', 'name', 'country_of_citizenship', 'height_in_cm', 'date_of_birth', 'position', 'image_url']]
df_players_mini['name'] = (df_players_mini['name']
                      .str.normalize('NFKD')
                      .str.encode('ascii', errors='ignore')
                      .str.decode('utf-8')
                      .str.lower()
                      .str.strip())

Create new `main_position` column with the first position listed for every player in a specific season and divide players in _Offense_ and _Defense_ roles

Rename positions for merging

In [6]:
top5_players['main_position'] = top5_players['pos_'].str.split(',').str[0]
top5_players['main_position'] = top5_players['main_position'].astype('str')

reemplazos = {
    'GK': 'Goalkeeper',
    'DF': 'Defender',
    'MF': 'Midfield',
    'FW': 'Attack'
}

top5_players['main_position'] = top5_players['main_position'].replace(reemplazos)

top5_players['role'] = np.where(
    top5_players['main_position'].isin(['Attack', 'Midfield']), 
    'Offense', 
    'Defense'
)


Eliminate players with no `date_of_birth` and take only the year for merging

In [7]:
df_players_mini = df_players_mini.dropna(subset=['date_of_birth'])
df_players_mini['date_of_birth'] = df_players_mini['date_of_birth'].astype(str).str[:4].astype(float)

Make left join of Fbref dataset with Transfermakt record on `name`, `position` and `date_of_birth`

In [8]:
df_players_merged = pd.merge(
    top5_players, 
    df_players_mini, 
    left_on=['name', 'main_position', 'born_'],  
    right_on=['name', 'position', 'date_of_birth'],    
    how='left'          
)
df_players_merged.isna().sum()

league                       0
season                       0
team                         0
player                       0
nation_                     10
                          ... 
country_of_citizenship    4174
height_in_cm              4111
date_of_birth             4063
position                  4063
image_url                 4063
Length: 71, dtype: int64

Remove players with no `player_id`

In [9]:
df_players_merged = df_players_merged.dropna(subset=['player_id'])

In [10]:
df_players_merged.isna().sum()

league                      0
season                      0
team                        0
player                      0
nation_                     6
                         ... 
country_of_citizenship    111
height_in_cm               48
date_of_birth               0
position                    0
image_url                   0
Length: 71, dtype: int64

Select relevant columns in player's valuations dataset, validate `date` data type and create `year` column

In [11]:
player_valuations = player_valuations[['player_id', 'date', 'market_value_in_eur']]

In [12]:
player_valuations['date'] = pd.to_datetime(player_valuations['date'])

player_valuations['year'] = player_valuations['date'].dt.year

Filter valuations to match years in Fbref dataset and players IDs for merging

In [13]:
df_valuations = player_valuations[player_valuations['player_id'].isin(df_players_merged['player_id'])]
df_valuations = df_valuations[(df_valuations['year'] >= 2017) & (df_valuations['year'] <= 2025)]

Select relevant columns in player's transfers dataset, validate `date` data type and create `year` column

In [14]:
transfers = transfers.drop(['from_club_id', 'to_club_id', 'player_name'], axis=1)

In [15]:
transfers['transfer_date'] = pd.to_datetime(transfers['transfer_date'])

transfers['year'] = transfers['transfer_date'].dt.year

Filter valuations to match years in Fbref dataset, remove null values and players IDs for merging

In [16]:
df_transfers = transfers[transfers['player_id'].isin(df_players_merged['player_id'])]
df_transfers = df_transfers[(df_transfers['year'] >= 2017) & (df_transfers['year'] <= 2025)]
df_transfers = df_transfers.dropna()

Group valuations by `player_id` and `year` to create columns with maximum and average valuation per year

In [17]:
df_valuations_grouped = df_valuations.groupby(['player_id', 'year']).agg(
    mean_market_value_in_eur=('market_value_in_eur', 'mean'),
    max_market_value_in_eur=('market_value_in_eur', 'max')
).reset_index()

Create `year` based on `season` column in Fbref dataset for merging

In [18]:
df_players_merged['year'] = df_players_merged['season'].astype(str).str[-2:].astype(int) + 2000

Inner join between Fbref dataset and valuations based on `player_id` and `year`

In [19]:
df_players = pd.merge(
    df_players_merged, 
    df_valuations_grouped, 
    on=['player_id', 'year'],
    how='inner'          
)

Remove redundant columns

In [20]:
df_players = df_players.drop(['nation_', 'pos_', 'born_', 'name', 'position', 'date_of_birth'], axis=1)

## Save dataframes to parquets

In [21]:
df_players.to_parquet('../Datasets/top5_players_stats.parquet', index=False)
df_transfers.to_parquet('../Datasets/top5_transfers.parquet', index=False)
df_valuations.to_parquet('../Datasets/top5_valuations.parquet', index=False)